# Train the shipped excitingness model

This notebook has one job: fit the model described in
`notes/model_history.md` § **Shipped model**, and save it to
`models/excitingness_model.joblib` for `predict_excitingness.py` to load
with `--model` — instead of that script silently retraining from scratch
on every single run.

It is **not** the model-comparison / feature-selection notebook — that's
`nbs/isdb_excitingness_model.ipynb`, and it stays the place for trying new
feature sets, bake-offs against other model classes, etc. This notebook
assumes those decisions are already made and just reproduces the winner:

- **Ridge regression**, `alpha=30`, vote-weighted, symmetric features only
- **11 features** (row 7, "+ team strength", in the feature-set history table)
- Trained on **World Cup matches** (WC2022 + WC2026), IMDb episode rating as
  the label, extra-time matches excluded (154 of 168 rows)

All feature engineering is imported directly from `predict_excitingness.py`
rather than reimplemented here — that's what guarantees training-time
features and inference-time features can never quietly drift apart.

## Setup — import the shipped config from `predict_excitingness.py`

In [ ]:
import sys, json, datetime
from pathlib import Path

sys.path.insert(0, "..")  # repo root, so we can import the script as a module
import predict_excitingness as pe

print(f"algorithm:     Ridge(alpha={pe.RIDGE_ALPHA})")
print(f"feature_set:   {len(pe.FEATURE_SET)} features")
for f in pe.FEATURE_SET:
    print(f"  - {f}")

## Load training data

Same loader `predict_excitingness.py` itself uses — builds `wc_labelled.csv` from raw shot JSON + IMDb ratings if it doesn't exist yet, then drops extra-time matches (they distort the 75-90+ window; see `EXCLUDE_EXTRA_TIME` in the script).

In [ ]:
data_dir = pe.find_data_dir()
pipeline_dir = pe.find_pipeline_dir(data_dir)
print(f"data dir: {data_dir}")

lab = pe.ensure_wc_labels(data_dir, pipeline_dir)
print(f"\ntraining rows: {len(lab)}")

## Build the feature matrix

In [ ]:
wc_cache = {}
for _, r in lab.iterrows():
    p = data_dir / f"xg_timeline/wc/wc{int(r.season_year)}_match_{r.match_id}.json"
    wc_cache[r.match_id] = pe.wc_shots(json.load(open(p))) if p.exists() else []

missing = sum(1 for v in wc_cache.values() if not v)
if missing:
    print(f"warning: {missing}/{len(wc_cache)} training matches have no raw shot file "
          f"on disk — their shot-derived features will be zero, which will pull "
          f"the fitted coefficients away from notes/model_history.md's documented "
          f"values. Make sure data/xg_timeline/wc/ is fully populated before "
          f"trusting this run's output.")

wc_maps = {yr: pe.pct_map(d) for yr, d in pe.FIFA_RANK.items()}
X_wc = pe.build_matrix(lab, lambda r: pe.shot_features(wc_cache[r.match_id]), wc_maps)
X_wc[pe.FEATURE_SET].describe().T[["mean", "std", "min", "max"]]

## Fit the model

Identical to the training block in `predict_excitingness.py`'s `main()` — vote-weighted (higher-vote-count IMDb ratings count for more), standardized features, ridge alpha 30.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge

y = lab.imdb_rating.values
w = (lab.imdb_votes.astype(float) / lab.imdb_votes.mean()).values

model = Pipeline([
    ("scale", StandardScaler()),
    ("ridge", Ridge(alpha=pe.RIDGE_ALPHA)),
])
model.fit(X_wc[pe.FEATURE_SET].values, y, ridge__sample_weight=w)
print(f"fitted on {len(lab)} matches, {len(pe.FEATURE_SET)} features")

## Sanity check against `notes/model_history.md`

The doc's "Shipped model" table is the source of truth for what this model is *supposed* to look like. If the raw WC shot data on disk is complete, these standardized coefficients should come out close to that table (small differences from CV-seed/refit noise are fine — a coefficient with the **wrong sign**, or one that collapsed to ~0 when the doc shows it as a real contributor, means something's missing upstream (usually incomplete raw shot files), not that the doc is wrong.

In [ ]:
documented = {
    "total_goals": 0.395, "avg_strength": -0.352, "upset": 0.202,
    "chasing_xg": 0.200, "big_chances_60-75": 0.180, "final5_swing_count": 0.145,
    "xg_absdiff_30-45": 0.117, "xg_absdiff_75-90plus": -0.060,
    "xg_absdiff_0-15": 0.048, "gap_strength": -0.039, "goal_diff_abs": -0.034,
}

ridge = model.named_steps["ridge"]
fitted = dict(zip(pe.FEATURE_SET, ridge.coef_.round(3)))

print(f"{'feature':<22}{'this run':>10}{'documented':>12}{'sign match':>12}")
for feat in pe.FEATURE_SET:
    a, b = fitted[feat], documented.get(feat, float('nan'))
    sign_ok = "OK" if (a * b) > 0 or abs(a) < 0.02 else "!! CHECK"
    print(f"{feat:<22}{a:>10.3f}{b:>12.3f}{sign_ok:>12}")

## Save the model

Writes the same bundle schema `predict_excitingness.py --save-model` itself produces (`build_model_bundle()`) — fitted model, feature set, and metadata describing the algorithm, training data, and coefficients, so the `.joblib` file is self-documenting without needing this notebook or the doc open next to it.

In [ ]:
out_path = Path("../models/excitingness_model.joblib")
out_path.parent.mkdir(parents=True, exist_ok=True)

import joblib
bundle = pe.build_model_bundle(model, len(lab))
joblib.dump(bundle, out_path)

print(f"saved -> {out_path}\n")
print(json.dumps(bundle["meta"], indent=2))

## Use it

```bash
python3 predict_excitingness.py --model models/excitingness_model.joblib --fetch
```

`--model` skips retraining entirely and scores straight from this saved fit. Re-run this notebook (and re-point `--model`) only when the shipped feature set or training data actually changes — not on every scoring run.